In [1]:
from build.models.base import *
from build import Genome

import torch

In [2]:
torch.set_printoptions(threshold=10)

In [3]:
device = 'cuda'

In [4]:
genomes = {key: Genome(key) for key in range(1, 2)}

In [5]:
batch_size = 5
seq_len = 16
input_dims = 2
output_dims = 1
heads = 1

In [6]:
linear = Linear(input_dims, output_dims, True, device)
linear.update(genomes)
linear, linear.weights.shape

(Linear[NeatModule](inputs=2, outputs=1, bias=True), torch.Size([1, 2, 1]))

In [7]:
inputs = torch.ones(len(genomes), batch_size, input_dims, device=device)

In [14]:
linear.weights.data, linear.biases.data

(Parameter containing:
 tensor([[[1.],
          [1.]]], device='cuda:0'),
 Parameter containing:
 tensor([[2.]], device='cuda:0'))

In [9]:
inputs

tensor([[[1., 1.],
         [1., 1.],
         [1., 1.],
         [1., 1.],
         [1., 1.]]], device='cuda:0')

In [10]:
inputs.shape

torch.Size([1, 5, 2])

In [11]:
# %%timeit -r 10   -n 10
outputs = linear(inputs)
outputs.shape

torch.Size([1, 5, 1])

In [12]:
outputs

tensor([[[4.],
         [4.],
         [4.],
         [4.],
         [4.]]], device='cuda:0')

In [34]:
linear(inputs[0], keys=1, fetch=False)

tensor([[[[-1.1586, -1.1586, -1.1586,  ..., -1.1586, -1.1586, -1.1586]],

         [[-3.7870, -3.7870, -3.7870,  ..., -3.7870, -3.7870, -3.7870]],

         [[ 3.6336,  3.6336,  3.6336,  ...,  3.6336,  3.6336,  3.6336]],

         ...,

         [[-0.9035, -0.9035, -0.9035,  ..., -0.9035, -0.9035, -0.9035]],

         [[-0.7297, -0.7297, -0.7297,  ..., -0.7297, -0.7297, -0.7297]],

         [[ 2.1114,  2.1114,  2.1114,  ...,  2.1114,  2.1114,  2.1114]]],


        [[[ 2.4267,  2.4267,  2.4267,  ...,  2.4267,  2.4267,  2.4267]],

         [[ 0.5445,  0.5445,  0.5445,  ...,  0.5445,  0.5445,  0.5445]],

         [[ 2.8910,  2.8910,  2.8910,  ...,  2.8910,  2.8910,  2.8910]],

         ...,

         [[ 2.7467,  2.7467,  2.7467,  ...,  2.7467,  2.7467,  2.7467]],

         [[-1.6210, -1.6210, -1.6210,  ..., -1.6210, -1.6210, -1.6210]],

         [[ 2.5651,  2.5651,  2.5651,  ...,  2.5651,  2.5651,  2.5651]]],


        [[[ 2.9686,  2.9686,  2.9686,  ...,  2.9686,  2.9686,  2.9686]],

    

Working Principle - Norm

In [35]:
affine = True

In [36]:
norm = LayerNorm((output_dims,), elementwise_affine=affine, bias=True, device=device)
norm.update(genomes)
norm

LayerNorm[NeatModule](shape=(32,), eps=1e-08, elementwise_affine=True, bias=True)

In [37]:
norm(outputs)[0, 0, 0, 0]

tensor([1., 1., 1.,  ..., 1., 1., 1.], device='cuda:0')

In [38]:
group_norm = GroupNorm(heads, output_dims, affine=affine, device=device)
group_norm.update(genomes)
group_norm

GroupNorm[NeatModule](num_groups=1, num_channels=32, eps=1e-08, affine=True)

In [39]:
group_norm(outputs, permute=(0, -2, -1, 1, 2))[0, 0, 0, 0]

tensor([1., 1., 1.,  ..., 1., 1., 1.], device='cuda:0')

In [40]:
rms_norm = RMSNorm(output_dims, elementwise_affine=affine, device=device)
rms_norm.update(genomes)
rms_norm

RMSNorm[NeatModule](shape=(32,), eps=1e-08, elementwise_affine=True)

In [41]:
rms_norm(outputs)[0, 0, 0, 0]

tensor([0., 0., 0.,  ..., 0., 0., 0.], device='cuda:0')

Timings

In [42]:
%%timeit -r 100 -n 5
norm(outputs)[0, 0, 0, 0];

219 μs ± 68.9 μs per loop (mean ± std. dev. of 100 runs, 5 loops each)


In [43]:
%%timeit -r 100 -n 5
group_norm(outputs, permute=(0, -2, -1, 1, 2))[0, 0, 0, 0];

189 μs ± 32.2 μs per loop (mean ± std. dev. of 100 runs, 5 loops each)


In [44]:
%%timeit -r 100 -n 5
rms_norm(outputs)[0, 0, 0, 0];

141 μs ± 32.4 μs per loop (mean ± std. dev. of 100 runs, 5 loops each)
